# exp002_drift_minimal inference

Notebook-first inference for a minimal residual model on top of the last-known TVT anchor.


## Contents

1. Setup and configuration
2. Train residual model and generate submission


## 1. Setup and configuration


In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import pandas as pd

from baseline import (
    HORIZONTAL_SUFFIX,
    build_drift_feature_frame,
    config_get,
    drift_strategy,
    fit_drift_model_from_files,
    optional_positive_int,
    predict_drift,
    primary_strategy,
    well_id_from_path,
)
from settings import EXPERIMENT_NAME, ExperimentPaths, load_config

DEBUG = os.environ.get("EXPERIMENT_DEBUG", "0") == "1"
MAX_WELLS_ENV = os.environ.get("EXPERIMENT_MAX_WELLS")
MAX_WELLS = int(MAX_WELLS_ENV) if MAX_WELLS_ENV else None

paths = ExperimentPaths()
paths.require_kaggle_runtime()
paths.ensure_output_dirs()
config = load_config()
primary = primary_strategy(config)
drift_name = drift_strategy(config)

print("Experiment:", EXPERIMENT_NAME)
print("Root:", paths.root)
print("Train data:", paths.train_data_dir)
print("Test data:", paths.test_data_dir)
print("Sample submission:", paths.sample_submission_path)
print("Submission path:", paths.submission_path)
print("Primary strategy:", primary)
print("Drift strategy:", drift_name)
print("Debug:", DEBUG, "Max wells:", MAX_WELLS)


## 2. Train residual model and generate submission


In [ ]:
def train_files_for_model(paths: ExperimentPaths, debug: bool, max_wells: int | None) -> list[Path]:
    files = sorted(paths.train_data_dir.glob(f"*{HORIZONTAL_SUFFIX}"))
    if not files:
        raise FileNotFoundError(f"no train horizontal well CSVs found in {paths.train_data_dir}")
    if debug:
        limit = max_wells
        if limit is None:
            limit = int(config.get("runtime", {}).get("debug_n_wells", 30))
        files = files[:limit]
    elif max_wells is not None:
        files = files[:max_wells]
    return files


def test_files(paths: ExperimentPaths) -> list[Path]:
    files = sorted(paths.test_data_dir.glob(f"*{HORIZONTAL_SUFFIX}"))
    if not files:
        raise FileNotFoundError(f"no test horizontal well CSVs found in {paths.test_data_dir}")
    return files


def configured_row_cap(key: str) -> int | None:
    return optional_positive_int(config_get(config, key, None))


if not paths.sample_submission_path.exists():
    raise FileNotFoundError(f"sample submission not found: {paths.sample_submission_path}")

model_files = train_files_for_model(paths, DEBUG, MAX_WELLS)
model, n_train_rows = fit_drift_model_from_files(
    model_files,
    config,
    seed=int(config["validation"]["seed"]),
    max_rows_total=configured_row_cap("model.training.max_train_rows_final"),
    max_rows_per_well=configured_row_cap("model.training.max_train_rows_per_well"),
)
print(f"Fitted drift model on {n_train_rows} sampled rows from {len(model_files)} wells")

predictions: dict[str, float] = {}
well_summaries: list[dict[str, object]] = []
for path in test_files(paths):
    well_id = well_id_from_path(path)
    df = pd.read_csv(path)
    frame = build_drift_feature_frame(df, config, include_target=False)
    if primary == drift_name:
        y_pred = predict_drift(frame, model, config)
    elif primary == "last_anchor":
        y_pred = frame.baseline_prediction
    else:
        raise ValueError(f"unsupported inference strategy: {primary}")

    for row_index, value in zip(frame.eval_indices, y_pred, strict=True):
        predictions[f"{well_id}_{int(row_index)}"] = float(value)
    well_summaries.append(
        {
            "well_id": well_id,
            "n_rows": int(len(df)),
            "n_eval": int(frame.eval_indices.size),
            "last_known_index": frame.last_known_index,
            "last_known_tvt": frame.last_known_tvt,
            "recent_slope": frame.recent_slope,
            "strategy": primary,
        }
    )

if well_summaries:
    pd.DataFrame(well_summaries).to_csv(
        paths.artifacts_dir / "inference_well_summaries.csv", index=False
    )

sample_submission = pd.read_csv(paths.sample_submission_path)
id_column = config["data"]["id_column"]
target_column = config["data"]["submission_target_column"]
missing_ids = sorted(set(sample_submission[id_column]) - set(predictions))
if missing_ids:
    preview = ", ".join(missing_ids[:5])
    raise ValueError(f"missing predictions for {len(missing_ids)} sample ids: {preview}")

output = sample_submission.copy()
output[target_column] = output[id_column].map(predictions).astype(float)
submission_path = paths.submission_path
output.to_csv(submission_path, index=False)
print("Created Kaggle submission:", submission_path)
print("Predicted rows:", len(predictions))
